## Vector Stores and Retrievers

Vector Stores — Definition

A vector store is a system that stores data as numerical vectors (embeddings) and allows efficient similarity search based on distance (e.g., cosine similarity or Euclidean distance).

In simple terms:
It converts text (or images, audio, etc.) into vectors and stores them so you can later find semantically similar content.

Why it matters:
Vector stores make it possible to search by meaning, not keywords.

Retrievers — Definition

A retriever is a component that queries a vector store and returns the most relevant stored items for a given input query.

In simple terms:
The retriever decides what to fetch from the vector store when a question is asked.

Why it matters:
Retrievers control relevance, filtering, and ranking—directly affecting the quality of results returned to an LLM.

Relationship between them (one-line)

A vector store stores embeddings, and a retriever pulls the most relevant ones for a query.


In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions. Known for thier loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent animals. They often enjoy solitude and quiet environments.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Birds can be wonderful pets. They are known for their ability to mimic sounds and their vibrant colors.",
        metadata={"source": "avian-pets-doc"},
    ),
    Document(
        page_content="Fish are low-maintenance pets. They require a properly set up aquarium to thrive.",
        metadata={"source": "aquatic-pets-doc"},
    ),
    Document(
        page_content="Hamsters are small rodents that make great pets for children. They are nocturnal and enjoy burrowing.",
        metadata={"source": "rodent-pets-doc"},
    )
]

In [4]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. Known for thier loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
 Document(metadata={'source': 'avian-pets-doc'}, page_content='Birds can be wonderful pets. They are known for their ability to mimic sounds and their vibrant colors.'),
 Document(metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.'),
 Document(metadata={'source': 'rodent-pets-doc'}, page_content='Hamsters are small rodents that make great pets for children. They are nocturnal and enjoy burrowing.')]

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = groq_api_key
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm= ChatGroq(groq_api_key=groq_api_key,model='llama-3.3-70b-versatile')
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x33c3df850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x32b90f610>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [20]:
## Vector Store Integration Example
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore


In [21]:
vectorstore.similarity_search("cat")

[Document(id='42b1a2c8-3274-4d83-86f3-e3963454593a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
 Document(id='74697c98-88d8-42ab-b880-5beb9d82d34c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
 Document(id='b64dca3a-3b4d-4077-ab97-029e97b65da0', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.'),
 Document(id='0f167b78-1f38-4249-9a29-e3672a913d5a', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.')]

In [22]:
## Async query example
await vectorstore.asimilarity_search("cat")

[Document(id='42b1a2c8-3274-4d83-86f3-e3963454593a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
 Document(id='74697c98-88d8-42ab-b880-5beb9d82d34c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
 Document(id='b64dca3a-3b4d-4077-ab97-029e97b65da0', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.'),
 Document(id='0f167b78-1f38-4249-9a29-e3672a913d5a', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.')]

In [23]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='42b1a2c8-3274-4d83-86f3-e3963454593a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
  1.0253400802612305),
 (Document(id='74697c98-88d8-42ab-b880-5beb9d82d34c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.'),
  1.0253400802612305),
 (Document(id='b64dca3a-3b4d-4077-ab97-029e97b65da0', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.'),
  1.507490634918213),
 (Document(id='0f167b78-1f38-4249-9a29-e3672a913d5a', metadata={'source': 'aquatic-pets-doc'}, page_content='Fish are low-maintenance pets. They require a properly set up aquarium to thrive.'),
  1.507490634918213)]

In [24]:
## Retriever Example
from typing import List, Tuple

from langchain_core.documents import Document
from langchain_core.runnables import  RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["dog", "cat"])

[[Document(id='987aa160-d15f-4c8d-ba0a-a726d056f68b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. Known for thier loyalty and friendliness.')],
 [Document(id='74697c98-88d8-42ab-b880-5beb9d82d34c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.')]]

In [25]:
## Other methods
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)
retriever.batch(["dog", "cat"])

[[Document(id='987aa160-d15f-4c8d-ba0a-a726d056f68b', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions. Known for thier loyalty and friendliness.')],
 [Document(id='74697c98-88d8-42ab-b880-5beb9d82d34c', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent animals. They often enjoy solitude and quiet environments.')]]

In [26]:
## RAG Example

from langchain_core.prompts import  ChatPromptTemplate
from langchain_core.runnables import  RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

reg_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response=reg_chain.invoke("Tell me about dogs.")
print(response.content)


According to the provided context, dogs are great companions, known for their loyalty and friendliness.
